In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
llm_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")


def rag_answer(query):
    # 1. retrieval
    candidate_docs = dense_search(query)

    # 2. reranking
    reranked_docs = rerank(query, candidate_docs)

    # 3. берём лучший документы
    context = "\n\n".join(reranked_docs[:1])

    # 4. формируем prompt
    prompt = f"""
    Answer ONLY using the context below.
    If the answer is not in the context, say "I don't know".

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    # 5. генерация
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = llm_model.generate(**inputs, max_new_tokens=100)

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

def rag_eval(num_samples=20):
    total_correct = 0

    answered_correct = 0
    answered_total = 0

    for i in range(num_samples):
        query = queries[i]
        true_answer = dataset[i]['answers']['text'][0]

        pred_answer = rag_answer(query)

        pred_lower = pred_answer.lower()
        true_lower = true_answer.lower()

        is_correct = true_lower in pred_lower
        is_answered = "i don't know" not in pred_lower

        # общая accuracy
        if is_correct:
            total_correct += 1

        # accuracy только по ответам
        if is_answered:
            answered_total += 1
            if is_correct:
                answered_correct += 1

        print("\n---")
        print("Q:", query)
        print("GT:", true_answer)
        print("Pred:", pred_answer)

    print("\nOverall Accuracy:", total_correct / num_samples)

    if answered_total > 0:
        print("Answered Accuracy:", answered_correct / answered_total)
        print("Answer Rate:", answered_total / num_samples)
    else:
        print("No answers given.")


rag_eval()


KeyboardInterrupt: 